In [1]:
print("Hola")

Hola


In [2]:
import os
import sys
print(os.getcwd())

sys.path.append(r'c:\Users\jmass\Documents\GitHub\TFG_GEINF\src\data')
from DatasetGenerator import DatasetGenerator


c:\Users\jmass\Documents\GitHub\TFG_GEINF\notebooks\experiments


In [3]:
import pandas as pd

num_particules = 10000

dg = DatasetGenerator(num_particules)

In [4]:
rw_data = dg.generate_RandomWalk(random_params=True, num_passos=200)
rw_data.to_pickle('rw_200p_10000t_rp.pkl')

KeyboardInterrupt: 

In [ ]:
bm_data = dg.generate_BrownianMotion(random_params=True, num_passos=200)
bm_data.to_pickle('bm_200p_10000t_rp.pkl')

In [6]:
ctrw_data = dg.generate_ContinuousTimeRandomWalk(random_params=True, num_passos=200)
ctrw_data.to_pickle('ctrw_200p_10000t_rp.pkl')

In [7]:
fbm_data = dg.generate_FractionalBrownianMotion(random_params=True, num_passos=200)
fbm_data.to_pickle('fbm_200p_10000t_rp.pkl')

c:\Users\jmass\Documents\GitHub\TFG_GEINF\env\Lib\site-packages\fbm\fbm.py:171: UserWarning: Combination of increments n and Hurst value H invalid for Davies-Harte method. Reverting to Hosking method. Occurs when n is small and Hurst is close to 1. 
  warnings.warn(


In [8]:
lw_data = dg.generate_LevyWalks(random_params=True, num_passos=200)
lw_data.to_pickle('lw_200p_10000t_rp.pkl')

In [9]:
attm_data = dg.generate_AnnealedTransitTimeModel(random_params=True, num_passos=200)
attm_data.to_pickle('attm_200p_10000t_rp.pkl')

In [10]:
sbm_data = dg.generate_ScaledBrownianMotion(random_params=True, num_passos=200)
sbm_data.to_pickle('sbm_200p_10000t_rp.pkl')

In [ ]:
import pandas as pd
import pickle

with open('./attm_200p_10000t_rp.pkl', 'rb') as file:
    attm_data = pickle.load(file)
print("attm_data completed")
with open('ctrw_200p_10000t_rp.pkl', 'rb') as file:
    ctrw_data = pickle.load(file)
print("ctrw_data completed")
with open('fbm_200p_10000t_rp.pkl', 'rb') as file:
    fbm_data = pickle.load(file)
print("fbm_data completed")
with open('lw_200p_10000t_rp.pkl', 'rb') as file:
    lw_data = pickle.load(file)
print("lw_data completed")
with open('rw_200p_10000t_rp.pkl', 'rb') as file:
    rw_data = pickle.load(file)
print("rw_data completed")
with open('sbm_200p_10000t_rp.pkl', 'rb') as file:
    sbm_data = pickle.load(file)
print("sbm_data completed")
with open('tb_200p_10000t_rp.pkl', 'rb') as file:
    bm_data = pickle.load(file)
print("bm_data completed")

data = pd.concat([attm_data, ctrw_data, fbm_data, lw_data, rw_data, sbm_data, bm_data], ignore_index=True)
data = data.drop(columns=['num_steps', 'attributes'])
data.to_pickle('all_types_200p_70000t_rp.pkl')

FileNotFoundError: [Errno 2] No such file or directory: './attm_200p_10000t_rp.pkl'

In [4]:
import pickle
with open('../../data/generated/all_types_200p_70000t_rp.pkl', 'rb') as file:
    data = pickle.load(file)

data = data.sample(frac=1).reset_index(drop=True)
data

,coords,trajectory_type
0,"[[0.0, -0.027688469342838643, -0.0448691634277...",Fractional Brownian Motion
1,"[[-0.5684039073565686, 0.5390483539845995, 0.9...",Lévy Walk
2,"[[0.0, -0.05076348206937256, -0.05490309225672...",Fractional Brownian Motion
3,"[[0.55445807568727, -1.7127226784875123, -1.83...",Lévy Walk
4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0,...",Random Walk
...,...,...
69995,"[[0.0, -0.9213666987043115, -1.110532669628362...",Brownian Motion
69996,"[[0.0, -0.009352313945581572, -0.0077629638120...",Fractional Brownian Motion
69997,"[[0.0, -0.09003346386365298, 0.894659153856894...",Brownian Motion
69998,"[[0.0, 0.0, -1.0, -1.0, -2.0, -3.0, -3.0, -2.0...",Random Walk


In [9]:
print(data.columns)

Index(['coords', 'trajectory_type'], dtype='object')


In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

In [6]:
valores_unicos = data['trajectory_type'].unique()
print(valores_unicos)

['Fractional Brownian Motion' 'Lévy Walk' 'Random Walk' 'ATTM'
 'Brownian Motion' 'CTRW' 'Scaled Brownian Motion']


In [7]:
# 1. Convertir las coordenadas a tensor 3D
# Asumiendo que 'data' es tu DataFrame
coords_list = [np.stack([row[0], row[1]], axis=1) for row in data['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32)

In [8]:
# 2. Convertir las etiquetas a numéricas
le = LabelEncoder()
labels = le.fit_transform(data['trajectory_type'])
labels_tensor  = torch.tensor(labels, dtype=torch.long)
y_tensor = F.one_hot(labels_tensor, num_classes=7).float()

In [9]:
# 3. Reorganizar dimensiones para Conv1D (n_samples, channels, seq_length)
X_tensor = X_tensor.permute(0, 2, 1)  # Nueva forma: (n, 2, m)

In [10]:
# 4. Dividir datos en entrenamiento y prueba (ejemplo simple)
train_size = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_test = y_tensor[:train_size], y_tensor[train_size:]

In [11]:
# 5. Crear DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# 6. Definir la arquitectura CNN
class TrajectoryCNN(nn.Module):
    def __init__(self, input_channels, seq_length, num_classes):
        super().__init__()
        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))
        self.fc1 = nn.Linear(4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    
    def forward(self, data):
        loop = 2
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.view(c.size(0), -1)  # Aplanar la salida
        c = F.relu(self.fc1(c))
        c = self.fc2(c)
        return c

In [18]:
# 7. Inicializar el modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)

In [19]:
model.first_conv.weight.shape

torch.Size([32, 2, 3])

In [20]:
# 8. Configurar entrenamiento
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [22]:
# 9. Entrenar el modelo
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")


Epoch [1/10], Loss: 0.7045
Epoch [2/10], Loss: 0.6001
Epoch [3/10], Loss: 0.5652
Epoch [4/10], Loss: 0.5474
Epoch [5/10], Loss: 0.5260
Epoch [6/10], Loss: 0.5108
Epoch [7/10], Loss: 0.4857
Epoch [8/10], Loss: 0.4530
Epoch [9/10], Loss: 0.4262
Epoch [10/10], Loss: 0.3982


In [35]:
# 10. Evaluación (opcional)
with torch.no_grad():
    test_outputs = model(X_test)
    _, predicted = torch.max(test_outputs, 1)  # Índices de las clases predichas
    true_labels = torch.argmax(y_test, dim=1)  # Convertir one-hot a índices
    accuracy = (predicted == true_labels).float().mean()
    print(f'Test Accuracy: {accuracy.item():.4f}')

Test Accuracy: 0.8131


In [ ]:
import torch
import dill
import inspect

# Guardar el model fora d'un notebook
# inspect.getsource(TrajectoryCNN)

# Funció necessària per poder uardar el model dins d'un notebook
def get_class_def_from_notebook(class_name: str):
    from IPython import get_ipython
    shell = get_ipython()
    for cell in shell.history_manager.get_range():
        if class_name in cell[2]:
            return cell[2]

get_class_def_from_notebook("TrajectoryCNN")


'# 6. Definir la arquitectura CNN\nclass TrajectoryCNN(nn.Module):\n    def __init__(self, input_channels, seq_length, num_classes):\n        super().__init__()\n        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)\n        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)\n        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)\n        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)\n        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)\n        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)\n        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)\n        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)\n        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)\n        self.max_pool = nn.MaxPool1d(2)\n        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))\n        self.fc1 = nn.Linear(4, 256)\n        self.fc2 = nn.Linear(256, num_classes)\n\n    \n  

In [ ]:
import dill

def save_model_with_source(model_instance, filename="model.pth"):
    torch.save({
        "model_name": model_instance.__class__.__name__,
        "model_type": "type_classifier",
        "model_source": get_class_def_from_notebook(model_instance.__class__.__name__), # Guardar el model fora d'un notebook: inspect.getsource(model_instance.__class__),
        "state_dict": model_instance.state_dict(),
        "attributes": {
            "input_channels": 2,
            "seq_length": X_tensor.shape[2],
            "num_classes": len(le.classes_)
        },
        "input_channels": 2
    }, filename, pickle_module=dill)

save_model_with_source(model, filename="model.pth")

In [42]:
import torch
import dill

def load_dynamic_model(model_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(model_path, map_location=device, pickle_module=dill)
    
    # Ejecutar el código fuente para recrear la clase
    exec(checkpoint["model_source"], globals())  # La clase se define en el scope global
    
    # Instanciar el modelo con sus argumentos originales
    model_class = globals()[checkpoint["model_source"].split("class ")[1].split("(")[0].strip()]
    model = model_class(**checkpoint["attributes"])
    model.load_state_dict(checkpoint["state_dict"])
    model.to(device)
    model.eval()
    return model

# Uso
model = load_dynamic_model("model.pth")
print(model)  # Muestra la estructura del modelo cargado

TrajectoryCNN(
  (first_conv): Conv1d(2, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv1_32): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2_32): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv_32_64): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv1_64): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2_64): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv_64_128): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv1_128): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2_128): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (max_pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (avg_pool): AdaptiveAvgPool2d(output_size=(1, 4))
  (fc1): Linear(in_features=4, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=7, bias=True)
)


In [46]:
# Seleccionar el primer ejemplo de X_test y añadir una dimensión batch
first_example = X_test[0].unsqueeze(0)  # shape: (1, 2, 200)

# Pasar por el modelo
model.eval()
with torch.no_grad():
    output = model(first_example)
    predicted_class = torch.argmax(output, dim=1).item()

# Mostrar el resultado
print("Predicción (índice de clase):", predicted_class)
print("Clase predicha:", le.inverse_transform([predicted_class])[0])
# Mostrar la clase real
true_class = torch.argmax(y_test[0]).item()
print("Clase real:", le.inverse_transform([true_class])[0])

Predicción (índice de clase): 5
Clase predicha: Random Walk
Clase real: Random Walk


In [49]:
model.__class__.__name__

'TrajectoryCNN'

In [27]:
# Guardar solo los parámetros del modelo
torch.save(model.state_dict(), 'trajectory_cnn_model_2.pth')

# Si deseas guardar el modelo completo
torch.save(model, 'trajectory_cnn_model_complete_2.pth')

In [ ]:
# Cargar solo los parámetros del modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)
model.load_state_dict(torch.load('trajectory_cnn_model_2.pth'))
model.eval()

# Si deseas cargar el modelo completo
# model = torch.load('trajectory_cnn_model_complete.pth')
# model.eval()